# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # Load .env file

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Create OpenAI client (it automatically uses the API key from the environment)
client = OpenAI()

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To reach yonder side, my dear interlocutor!


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How can I assist you today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to personal information like your name. You can tell me your name if you'd like!


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [8]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [9]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'fa62247d-1841-4d05-92b2-0b00a871f97a': {'version…

In [10]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large",
    "price": 12.95
  },
  "toppings": [
    {
      "type": "extra cheese",
      "price": 2.00
    },
    {
      "type": "mushrooms",
      "price": 1.50
    }
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium",
      "price": 2.00
    }
  ],
  "sides": [],
  "total price": 18.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Exercide 1 AreperaBot

1. Start conversation context with a system message:
      - define that the bot is ArepaBot
      - define that it works in a Venezuelan arepa restaurant
      - include menu and behavior rules

2. When the user sends a message:
      - add user message to context
      - send full context to the LLM
      - get assistant response
      - add assistant response to context
      - display both messages in the chat UI

3. After the order is complete:
      - send a final system message asking for a JSON summary
      - LLM returns structured order summary

In [11]:
!pip install jupyter_bokeh

In [12]:
def collect_messages(_):
    # Read user input
    prompt = inp.value_input
    inp.value = ''

    # Add user message to context
    context.append({'role': 'user', 'content': prompt})

    # Get assistant response using full conversation context
    response = get_completion_from_messages(context)

    # Add assistant response to context
    context.append({'role': 'assistant', 'content': response})

    # Display messages in the Panel UI
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600,
                                              styles={'background-color': '#F6F6F6'}))
    )

    # Return updated chat history
    return pn.Column(*panels)

In [13]:
panels = [] # collect display 

context = [
    {
        "role": "system",
        "content": """
You are ArepaBot, an automated assistant for taking orders in a Venezuelan arepa restaurant.
Your tone is friendly, warm, and conversational.

Your workflow:
1. Greet the customer.
2. Ask what they would like to order.
3. Ask clarifying questions:
   - Type of arepa
   - Fillings
   - Extras
   - Drinks
   - Pickup or delivery
4. When the customer finishes, summarize the full order.
5. Ask if they want to add anything else.
6. If delivery, ask for the address.
7. Finally, collect the payment method.

MENU:
Arepas:
- Reina Pepiada: 6.50
- Pelua: 7.00
- Domino: 5.50
- Pabellon: 8.00
- Catira: 6.00
- Sifrina: 7.50

Extras:
- Yellow cheese: 1.00
- Shredded cheese: 1.00
- Avocado: 1.50
- Fried plantains: 1.50
- Bacon: 2.00

Drinks:
- Papelón con limón: 3.00
- Malta: 2.50
- Soda: 2.00
- Water: 1.50

Always respond in short, friendly sentences.
"""
    }
]

In [14]:
import panel as pn
pn.extension()

# List to store the chat display
panels = []

# Text input widget
inp = pn.widgets.TextInput(
    value="Hi",
    placeholder="Type your message here..."
)

# Button to send the message
button_conversation = pn.widgets.Button(name="Chat!")

# Bind the button to the collect_messages function
interactive_conversation = pn.bind(collect_messages, button_conversation)

# Build the dashboard layout
dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'84cf8075-c615-45df-bc28-94016f4f68f9': {'version…

In [15]:
messages = context.copy()

messages.append({
    "role": "system",
    "content": """
Create a JSON summary of the previous food order.
Include the following fields:
1) list of arepas with their prices
2) list of extras with their prices
3) list of drinks with their prices
4) total price
Make sure the JSON is valid and properly formatted.
"""
})

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "order": {
    "arepas": [
      {
        "Reina Pepiada": 6.50
      }
    ],
    "extras": [
      {
        "Avocado": 1.50
      }
    ],
    "drinks": [
      {
        "Papelón con Limón": 3.00
      }
    ],
    "total_price": 11.00
  }
}


### Conclusion

We built a complete conversational ordering system for a Venezuelan arepa restaurant. The project included defining a system prompt for ArepaBot, creating a message‑handling function to manage the conversation flow, and building an interactive Panel dashboard where the user can type messages and see responses. We also added a final step to generate a structured JSON summary of the customer’s order.

Even though the backend logic works correctly — as shown by the successful JSON output — the chat interface still doesn’t display the assistant’s responses. Instead, the Panel dashboard keeps showing a loading spinner. This means the model is responding, but the UI is not rendering the messages, so the conversation cannot continue inside the chat window.

In short: the bot works, but the Panel interface is not updating, which is why the chat appears to “think forever” without showing any reply.


## Exercise 2 – DentalBot (Appointment Scheduler)

- Start conversation context with a system message:
- define that the bot is DentalBot
- define that it works for a dental clinic
- include the appointment types, workflow, and behavior rules
- instruct the bot to collect:
- appointment type
- preferred date
- preferred time
- full name
- phone number
- When the user sends a message:
- add the user message to the conversation context
- send the full context to the LLM
- receive the assistant response
- add the assistant response to the context
- display both messages in the chat UI (user → assistant)
- After the appointment details are complete:
- send a final system message asking for a structured JSON summary
- LLM returns a JSON object containing:
- appointment type
- date
- time
- patient name
- phone number


In [16]:
def collect_messages(_):
    # Read user input
    prompt = inp.value_input
    inp.value = ''

    # Add user message to context
    context.append({'role': 'user', 'content': prompt})

    # Get assistant response using full conversation context
    response = get_completion_from_messages(context)

    # Add assistant response to context
    context.append({'role': 'assistant', 'content': response})

    # Display messages in the Panel UI
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600,
                                              styles={'background-color': '#F6F6F6'}))
    )

    # Return updated chat history
    return pn.Column(*panels)

In [17]:
panels = [] # collect display 
context = [
    {
        "role": "system",
        "content": """
You are DentalBot, an automated assistant for a dental clinic.
Your job is to help patients schedule appointments and answer basic questions.
Your tone is warm, professional, and reassuring.

Available appointment types:
- Dental cleaning
- General check-up
- Tooth pain / emergency
- Orthodontics consultation
- Whitening

Workflow:
1. Greet the patient.
2. Ask what type of appointment they need.
3. Ask for preferred date.
4. Ask for preferred time.
5. Ask for full name.
6. Ask for phone number.
7. Confirm the appointment.
8. At the end, produce a structured JSON summary.

Rules:
- Keep responses short and friendly.
- Never give medical diagnoses.
- If the patient describes symptoms, suggest booking an appointment.
- If the patient asks for prices, provide general ranges only.
"""
    }
]

In [18]:
import panel as pn
pn.extension()

panels = []

inp = pn.widgets.TextInput(
    value="Hi",
    placeholder="Type your message here..."
)

button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'bd152d1b-19b6-4fb3-b839-49e28a83ad9a': {'version…

In [23]:
context.append({
    "role": "system",
    "content": """
Now produce a JSON summary of the dental appointment.
The JSON must contain:
- appointment_type
- date
- time
- patient_name
- phone_number

Return ONLY valid JSON with no extra text.
"""
})
response = get_completion_from_messages(context, temperature=0)
print(response)

{
  "appointment_type": "General check-up",
  "date": "May 1st, 2026",
  "time": "To be confirmed",
  "patient_name": "Lourdes Lee",
  "phone_number": "348888888"
}


## Conclusion
In this exercise, we designed and implemented DentalBot, a conversational assistant for scheduling dental appointments. We began by defining a structured system prompt that establishes the bot’s role, tone, available appointment types, and the step‑by‑step workflow it must follow. This ensures the assistant consistently gathers all the necessary information from the patient.

We then implemented the conversation logic: each user message is added to the context, sent to the model, and the assistant’s response is appended and displayed. This mirrors the architecture used in the ArepaBot project and keeps the design modular and predictable.

Finally, we created the JSON Summary Generator, which produces a clean, structured summary of the appointment once all details are collected. The JSON includes the appointment type, date, time, patient name, and phone number—making it easy to store or integrate into a real scheduling system.

Although the Python chat interface experienced a visualization issue that prevented interactive responses from appearing in the UI, the model still processed the messages correctly. All the information requested by DentalBot was understood, stored in the conversation context, and successfully used to generate the final JSON summary.

Overall, the project now has a complete and functional conversational flow, from greeting the patient to producing a structured appointment summary.


## Exercise 3 – LaundryBot (Home Pickup Laundry Service)

- Start conversation context with a system message:
- Define that the bot is LaundryBot.
- Specify that it works for a home‑pickup laundry service.
- Explain that the bot must collect:
- full home address
- pickup date
- pickup time
- phone number
- optional notes from the customer
- Include a rule stating that the cost is 10 per kilo of laundry.
- Set a friendly, helpful tone focused on logistics, not opinions.
- When the user sends a message:
- Add the user message to the conversation context.
- Send the full context to the LLM.
- Receive the assistant response.
- Add the assistant response to the context.
- Display both messages in the chat UI (user → assistant).
- After all pickup details are collected:
- Send a final system message asking for a structured JSON summary.
- LLM returns a JSON object containing:
- address
- pickup_date
- pickup_time
- phone_number
- notes
- price_per_kilo: 10


In [24]:
def collect_messages(_):
    # Read user input
    prompt = inp.value_input
    inp.value = ''

    # Add user message to context
    context.append({'role': 'user', 'content': prompt})

    # Get assistant response using full conversation context
    response = get_completion_from_messages(context)

    # Add assistant response to context
    context.append({'role': 'assistant', 'content': response})

    # Display messages in the Panel UI
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600,
                                              styles={'background-color': '#F6F6F6'}))
    )

    # Return updated chat history
    return pn.Column(*panels)

In [25]:
panels = []  # collect display

context = [
    {
        "role": "system",
        "content": """
You are LaundryBot, an automated assistant for a home‑pickup laundry service.
Your job is to help customers schedule a laundry pickup at their home.
Your tone is friendly, efficient, and clear.

Your workflow:
1. Greet the customer.
2. Ask for the full home address for the pickup.
3. Ask for the preferred pickup date.
4. Ask for the preferred pickup time.
5. Ask for the customer's phone number.
6. Ask if they want to add any notes (optional).
7. Confirm all the details back to the customer.
8. At the end, produce a structured JSON summary of the pickup request.

Information you must collect:
- full address
- pickup date
- pickup time
- phone number
- optional notes

Rules:
- Keep responses short and helpful.
- Do not ask for unnecessary information.
- Do not provide prices during the conversation.
- After the JSON summary is generated, include a final message stating:
  "The cost of the service is 10 per kilo of laundry."

JSON summary format:
{
  "address": "",
  "pickup_date": "",
  "pickup_time": "",
  "phone_number": "",
  "notes": "",
  "price_per_kilo_message": "The cost of the service is 10 per kilo of laundry."
}
"""
    }
]

In [26]:
import panel as pn
pn.extension()

panels = []

inp = pn.widgets.TextInput(
    value="Hi",
    placeholder="Type your message here..."
)

button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'854a6309-1bc6-480f-b2fc-6050d8ea72bf': {'version…

In [31]:
context.append({
    "role": "system",
    "content": """
Now produce a JSON summary of the laundry pickup request.
The JSON must contain:
- address
- pickup_date
- pickup_time
- phone_number
- notes
- price_per_kilo_message: "The cost of the service is 10 per kilo of laundry."

Return ONLY valid JSON with no extra text.
"""
})
response = get_completion_from_messages(context, temperature=0)
print(response)

{
  "address": "Plaza de Blass, floor 1, Door 4, Riva Vaciamadrid, Madrid",
  "pickup_date": "April 10",
  "pickup_time": "9.00 AM",
  "phone_number": "348888888",
  "notes": "",
  "price_per_kilo_message": "The cost of the service is 10 per kilo of laundry."
}


## Conclusion 

In this exercise, we developed LaundryBot, a conversational assistant designed to manage home laundry pickup requests. We began by defining a structured system prompt that guides the bot through a clear workflow: collecting the customer’s full address, pickup date, pickup time, phone number, and any optional notes. The assistant maintains a friendly and efficient tone, focusing strictly on the logistics of the service.

We implemented the conversation flow using a context-based approach. Each user message is appended to the conversation history, sent to the model, and the assistant’s response is added back into the context. This ensures consistency and coherence throughout the interaction, following the same architecture used in the previous exercises.

We also built the JSON Summary Generator, which produces a structured summary of the pickup request once all required information has been gathered. This summary includes the mandatory final message: “The cost of the service is 10 per kilo of laundry.” This makes the output easy to store, process, or integrate into a real scheduling system.

Although the Python chat interface experienced a visualization issue that prevented the messages from appearing in the UI, the model still processed and understood all user inputs correctly. The logic of the conversation worked as intended, and the final JSON summary was generated without any loss of information.

Overall, LaundryBot completes the set of conversational assistants with a clean, modular, and reproducible workflow suitable for demos, teaching, and real-world prototypes.
